# 04 · Train the pooled predictors (TwoRoom)

Trains one DINO-WM-style predictor per token budget on the features cached by notebook 03. Everything is
the full-grid reference's recipe except the number of tokens:

- predictor: `CausalPredictor` depth 6, 16 heads, MLP 2048, dropout 0.1, token dim 384 + 10 (action embedding)
- action encoder: linear map of the 5-step action block (10 values) to 10 dims, tiled onto every token
- history 3 frames, frameskip 5, teacher forcing with the causal frame mask
- loss: MSE between the predicted and the actual next-frame visual tokens (action dims excluded)
- AdamW, lr 5e-4, weight decay 0, gradient clipping 1.0 (the upstream DINO-WM recipe)

What is **not** the reference's: the data (the first 2,000 of 10,000 episodes) and the budget (a fixed
number of steps, the same for every variant). Every variant trains on exactly the same clips, so the
curve across token counts is a fair comparison; the absolute numbers are those of a small-budget model.

Checkpoints are `weights.pt` + `config.json` folders that `stable_worldmodel.wm.utils.load_pretrained`
rebuilds; notebook 05 plans with them through the same CEM as notebook 01.

## 1. Paths and environment variables

In [ ]:
import os
import shutil
import sys
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    ROOT = Path("/content")
else:
    ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# must be set before importing stable_worldmodel / mujoco
os.environ["STABLEWM_HOME"] = str(ROOT / "stablewm")      # datasets/ and checkpoints/ live here
os.environ.setdefault("WANDB_MODE", "disabled")
if sys.platform == "linux":
    os.environ.setdefault("MUJOCO_GL", "egl")             # headless rendering; macOS uses its default

HOME = Path(os.environ["STABLEWM_HOME"])
(HOME / "datasets").mkdir(parents=True, exist_ok=True)
print("root:", ROOT)
print("STABLEWM_HOME:", HOME)
print(f"free disk: {shutil.disk_usage(HOME).free / 1e9:.0f} GB")

## 2. Install (Colab only)

In [ ]:
PACKAGES = [
    "stable-worldmodel[train,env]==0.1.1",
    "hdf5plugin",
    "mujoco>=3.14,<3.15",
    "transformers>=4.50,<5",
    "datasets>=3",
    "huggingface-hub<1",
]

if IN_COLAB:
    import subprocess

    def pip(*args):
        cmd = [sys.executable, "-m", "uv", "pip", "install", "--system", *args]
        result = subprocess.run(cmd, capture_output=True, text=True)
        if result.returncode != 0:
            print((result.stdout + result.stderr)[-6000:])      # uv's own error message
            raise RuntimeError("install failed, see the output above")

    # Install system dependency swig required to compile box2d-py
    print("Installing system dependencies (swig)... ")
    subprocess.run(["apt-get", "update", "-y"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
    subprocess.run(["apt-get", "install", "-y", "swig"], check=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

    if sys.version_info >= (3, 13):
        stub = Path("/content/labmaze_stub")
        (stub / "labmaze").mkdir(parents=True, exist_ok=True)
        (stub / "labmaze" / "__init__.py").write_text("# empty stand-in, see notebook section 2\n")
        (stub / "pyproject.toml").write_text(
            "[build-system]\nrequires = [\"setuptools\"]\nbuild-backend = \"setuptools.build_meta\"\n"
            "[project]\nname = \"labmaze\"\nversion = \"1.0.6\"\n"
        )
        pip(str(stub))

    pip(*PACKAGES)
    print("installed on python", sys.version.split()[0])

## 3. The `dwt` package

The model definition lives in `src/dwt/models.py` of the project repo. On Colab this cell writes the same file so the notebook stays self-contained; the checkpoints' `config.json` refers to `dwt.models.PooledDino`, so notebook 05 needs it too.

In [ ]:
MODELS_SRC = r'''"""Pooled-token DINO-WM variants, built on stable_worldmodel's PreJEPA.

The frozen encoder is DINOv2 ViT-S/14 fed exactly as the full-grid reference feeds it (frames scaled
to [-1, 1], resized to 196 px, `forward_features`), followed by a fixed pooling of the 14 x 14 patch grid.
Only the token count differs between the variants and the reference:

    cls   1   CLS token
    mean  1   mean over the 196 patches
    g2    4   adaptive average pool to 2 x 2
    g4   16   adaptive average pool to 4 x 4
    g7   49   adaptive average pool to 7 x 7
    full 196  no pooling (the reference layout)

A model is `PreJEPA(encoder=PooledDino(variant), predictor=CausalPredictor(num_patches=k, ...))`, so
planning (`get_cost`), rollout and checkpoint loading (`load_pretrained`) come from stable_worldmodel.
"""
from types import SimpleNamespace

import torch
import torch.nn.functional as F
from torch import nn
from torchvision.transforms import v2 as transforms

from stable_worldmodel.wm.prejepa.module import CausalPredictor, Embedder
from stable_worldmodel.wm.prejepa.prejepa import PreJEPA

GRID = 14                     # 196 px / 14 px patches
FEAT_DIM = 384                # DINOv2 ViT-S
ENCODER_PX = GRID * 14
HUB_NAME = "dinov2_vits14"
TOKENS = {"cls": 1, "mean": 1, "g2": 4, "g4": 16, "g7": 49, "full": GRID * GRID}
_GRID_SIDE = {"g2": 2, "g4": 4, "g7": 7, "full": GRID}


def pixel_transform(img_size=224):
    """uint8 HWC frame -> float CHW in [-1, 1], as the reference's `default_transform`."""
    return transforms.Compose([
        transforms.ToImage(),
        transforms.ToDtype(torch.float32, scale=True),
        transforms.Resize(img_size),
        transforms.CenterCrop(img_size),
        transforms.Normalize([0.5, 0.5, 0.5], [0.5, 0.5, 0.5]),
    ])


def pool_tokens(cls_token, patches, variant):
    """cls_token (B, 384), patches (B, 196, 384) -> (B, TOKENS[variant], 384)."""
    if variant == "cls":
        return cls_token[:, None]
    if variant == "mean":
        return patches.mean(1, keepdim=True)
    side = _GRID_SIDE[variant]
    if side == GRID:
        return patches
    grid = patches.transpose(1, 2).reshape(patches.shape[0], FEAT_DIM, GRID, GRID)
    if grid.device.type == "mps":        # MPS cannot pool 14 -> 4 (non-divisible sizes)
        grid = grid.cpu()
    pooled = F.adaptive_avg_pool2d(grid, side).flatten(2).transpose(1, 2)
    return pooled.to(patches.device)


class PooledDino(nn.Module):
    """Frozen DINOv2 -> pooled tokens, returned the way PreJEPA reads HF encoders
    (`.last_hidden_state`, whose first token it drops as the CLS placeholder).

    The backbone is deliberately not registered as a submodule: it is frozen and public, so it stays out of
    `state_dict()` and checkpoints hold only the trained parts. It follows the input tensor's device.
    """

    def __init__(self, variant, hub_name=HUB_NAME):
        super().__init__()
        assert variant in TOKENS, f"unknown variant {variant!r}, expected one of {list(TOKENS)}"
        self.variant = variant
        self.hub_name = hub_name
        backbone = torch.hub.load("facebookresearch/dinov2", hub_name).eval()
        backbone.requires_grad_(False)
        self._backbone = [backbone]
        self.config = SimpleNamespace(hidden_size=FEAT_DIM)       # mirrors the HF attribute PreJEPA users read

    @property
    def backbone(self):
        return self._backbone[0]

    @property
    def num_tokens(self):
        return TOKENS[self.variant]

    @torch.no_grad()
    def forward(self, pixels, **kwargs):
        """pixels (B, 3, H, W) in [-1, 1] -> last_hidden_state (B, 1 + tokens, 384)."""
        backbone = self.backbone
        if next(backbone.parameters()).device != pixels.device:
            backbone.to(pixels.device)
            self._backbone = [backbone]
        x = F.interpolate(pixels, size=(ENCODER_PX, ENCODER_PX), mode="bilinear",
                          antialias=True, align_corners=False)
        out = backbone.forward_features(x)
        tokens = pool_tokens(out["x_norm_clstoken"], out["x_norm_patchtokens"], self.variant)
        return SimpleNamespace(last_hidden_state=torch.cat([out["x_norm_clstoken"][:, None], tokens], dim=1))


def model_config(variant, action_dim=2, frameskip=5, history=3, action_emb=10,
                 depth=6, heads=16, mlp_dim=2048, dim_head=64, dropout=0.1):
    """Hydra-instantiable config of a pooled DINO-WM; the predictor hyperparameters are the reference's."""
    return {
        "_target_": "stable_worldmodel.wm.PreJEPA",
        "history_size": history,
        "num_pred": 1,
        "interpolate_pos_encoding": False,
        "encoder": {"_target_": "dwt.models.PooledDino", "variant": variant},
        "predictor": {
            "_target_": "stable_worldmodel.wm.prejepa.module.CausalPredictor",
            "num_patches": TOKENS[variant], "num_frames": history, "dim": FEAT_DIM + action_emb,
            "depth": depth, "heads": heads, "mlp_dim": mlp_dim, "dim_head": dim_head,
            "dropout": dropout, "emb_dropout": 0.0,
        },
        "extra_encoders": {
            "_target_": "torch.nn.ModuleDict",
            "modules": {"action": {"_target_": "stable_worldmodel.wm.prejepa.module.Embedder",
                                   "in_chans": action_dim * frameskip, "emb_dim": action_emb}},
        },
    }


def build_model(variant, **kwargs):
    """Instantiate a pooled DINO-WM from `model_config` (same code path as `load_pretrained`)."""
    from hydra.utils import instantiate
    return instantiate(model_config(variant, **kwargs))


def trainable_parameters(model):
    """Predictor and action embedder; the encoder is frozen and lives outside the state dict anyway."""
    return [p for n, p in model.named_parameters() if n.startswith(("predictor.", "extra_encoders."))]
'''

INIT_SRC = r'''"""dwt: pooled-token DINO world models (token budget study)."""
'''

SRC_DIR = ROOT / "src"
if IN_COLAB:
    (SRC_DIR / "dwt").mkdir(parents=True, exist_ok=True)
    (SRC_DIR / "dwt" / "__init__.py").write_text(INIT_SRC)
    (SRC_DIR / "dwt" / "models.py").write_text(MODELS_SRC)
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

import dwt.models

print("dwt.models from", dwt.models.__file__)

## 4. Imports and settings

In [ ]:
import csv
import json
import platform
import time
import warnings

import h5py
import numpy as np
import torch
import torch.nn.functional as F
from omegaconf import OmegaConf

warnings.filterwarnings("ignore")
import stable_worldmodel as swm

from dwt.models import FEAT_DIM, TOKENS, build_model, model_config, trainable_parameters

DEVICE = "cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu"
GPU = torch.cuda.get_device_name(0) if DEVICE == "cuda" else DEVICE
SMOKE = DEVICE != "cuda"

TAG = "tworoom_local_ep20" if SMOKE else "tworoom_ep2000"     # feature files from notebook 03
VARIANTS = ["cls", "mean", "g2", "g4", "g7"]
HISTORY, FRAMESKIP = 3, 5
N_STEPS_CLIP = HISTORY + 1                                      # frames per training clip
STEPS = 30 if SMOKE else 30000                                  # optimizer steps per variant (about 4.5 h on a T4)
MILESTONES = [10, 20, 30] if SMOKE else [10000, 30000]          # snapshot folders kept at these steps
BATCH = 8 if SMOKE else 64
LR, WEIGHT_DECAY, GRAD_CLIP = 5e-4, 0.0, 1.0
VAL_FRACTION = 0.1                                              # last 10% of the cached episodes
EVAL_EVERY, CKPT_EVERY = (10, 10) if SMOKE else (500, 1000)
import psutil

RAM_LIMIT_GB = 0.6 * psutil.virtual_memory().available / 1e9    # feature files up to this size are held in RAM
SEED = 0

FEATURES_DIR = HOME / "features"
CKPT_DIR = HOME / "checkpoints"
DRIVE_DIR = Path("/content/drive/MyDrive/dino-wm-tokens") if IN_COLAB else None   # backups; None = no backup
LOG_DIR = ROOT / "results" / "training"
LOG_DIR.mkdir(parents=True, exist_ok=True)

print("python", platform.python_version(), "| torch", torch.__version__, "| device", GPU)
print("smoke configuration" if SMOKE else "full configuration", "|", TAG, "|", STEPS, "steps x batch", BATCH,
      f"| files up to {RAM_LIMIT_GB:.0f} GB held in RAM")

## 5. Feature files

On Colab the files come back from Google Drive to the session disk (reading straight off the mount is slow).

In [ ]:
FILES = {v: FEATURES_DIR / f"{TAG}_{v}.h5" for v in VARIANTS}
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    (DRIVE_DIR / "checkpoints").mkdir(parents=True, exist_ok=True)
missing = [v for v, p in FILES.items() if not p.exists()]
if missing and IN_COLAB:
    src = DRIVE_DIR / "features"
    FEATURES_DIR.mkdir(exist_ok=True)
    for v in missing:
        t0 = time.perf_counter()
        shutil.copy(src / FILES[v].name, FILES[v])
        print(f"copied {FILES[v].name} ({FILES[v].stat().st_size / 1e9:.2f} GB) in {time.perf_counter() - t0:.0f} s")
missing = [v for v, p in FILES.items() if not p.exists()]
assert not missing, f"feature files missing: {missing}; run notebook 03 first"

with h5py.File(FILES["cls"], "r") as f:
    EP_LEN, EP_OFFSET = f["ep_len"][:], f["ep_offset"][:]
    ACTIONS = f["action"][:].astype(np.float32)
    N_EPISODES, N_FRAMES = len(EP_LEN), len(ACTIONS)
    print(dict(f.attrs))
ACTION_DIM = ACTIONS.shape[1]
print(f"{N_EPISODES} episodes, {N_FRAMES} frames, action dim {ACTION_DIM}")

## 6. Clips, batches and the training step

A clip is 20 consecutive env steps: 4 observation frames at stride 5 and the 20 actions between them,
reshaped to 4 blocks of 10 values (as `stable_worldmodel`'s loader and the reference do it). The split is
by episode: the last 10% of episodes are validation.

Actions are z-scored with the mean and std of the cached episodes; the stats are written to the checkpoint.
The released dataset stores `nan` for actions at some steps (the end of an episode); those are excluded
from the statistics and replaced by zeros, exactly as `stable_worldmodel`'s training script does.

In [ ]:
SPAN = N_STEPS_CLIP * FRAMESKIP
nan_rows = np.isnan(ACTIONS).any(axis=1)
ACTION_MEAN, ACTION_STD = ACTIONS[~nan_rows].mean(0), ACTIONS[~nan_rows].std(0)
ACTIONS_Z = np.nan_to_num((ACTIONS - ACTION_MEAN) / ACTION_STD, nan=0.0).astype(np.float32)
print(f"{nan_rows.sum()} of {N_FRAMES} action rows are nan -> zero after scaling | mean {ACTION_MEAN.round(3)} std {ACTION_STD.round(3)}")
assert np.isfinite(ACTIONS_Z).all()

n_val = max(1, int(round(N_EPISODES * VAL_FRACTION)))
episodes = {"train": np.arange(N_EPISODES - n_val), "val": np.arange(N_EPISODES - n_val, N_EPISODES)}
clips = {split: np.array([(EP_OFFSET[e] + s) for e in eps for s in range(EP_LEN[e] - SPAN + 1)])
         for split, eps in episodes.items()}                       # global row index of each clip's first frame
print({split: f"{len(eps)} episodes, {len(clips[split])} clips" for split, eps in episodes.items()})

FRAME_OFFSETS = np.arange(N_STEPS_CLIP) * FRAMESKIP                 # 0, 5, 10, 15


def rechunk_rows(path):
    """Rewrite a feature file with one frame per HDF5 chunk, for random-access reads.

    Notebook 03 writes 256 frames per chunk, which is right for sequential writing but makes a random
    training batch read whole chunks for single rows (gigabytes per step). Done once per file.
    """
    tmp = path.with_suffix(".rowchunks.h5")
    with h5py.File(path, "r") as src, h5py.File(tmp, "w") as dst:
        feat = src["feat"]
        if feat.chunks is not None and feat.chunks[0] == 1:
            tmp.unlink()
            return
        print(f"rechunking {path.name} ({path.stat().st_size / 1e9:.1f} GB) for random access...", end=" ", flush=True)
        t0 = time.perf_counter()
        out = dst.create_dataset("feat", shape=feat.shape, dtype=feat.dtype, chunks=(1, *feat.shape[1:]))
        for i in range(0, feat.shape[0], 4096):
            out[i:i + 4096] = feat[i:i + 4096]
        for key in src.keys():
            if key != "feat":
                dst.create_dataset(key, data=src[key][:])
        dst.attrs.update(src.attrs)
        print(f"done in {time.perf_counter() - t0:.0f} s")
    tmp.replace(path)


class Features:
    """Rows of one variant's feature file, in RAM when the file is small enough, else gathered from HDF5."""

    def __init__(self, path):
        self.in_ram = path.stat().st_size <= RAM_LIMIT_GB * 1e9
        if not self.in_ram:
            rechunk_rows(path)
        self.f = h5py.File(path, "r")
        self.feat = self.f["feat"][:] if self.in_ram else self.f["feat"]
        self.tokens = self.feat.shape[1]

    def clip_batch(self, starts):
        """starts (B,) -> frames (B, 4, tokens, 384) float32, actions (B, 4, 10) float32 on DEVICE."""
        rows = (starts[:, None] + FRAME_OFFSETS[None]).reshape(-1)                 # (B*4,)
        if self.in_ram:
            frames = self.feat[rows]
        else:
            unique_rows, inverse = np.unique(rows, return_inverse=True)             # h5py needs strictly increasing indices
            frames = self.feat[unique_rows][inverse]                                # (clips 5 steps apart share frames)
        frames = torch.from_numpy(frames.astype(np.float32)).view(len(starts), N_STEPS_CLIP, self.tokens, FEAT_DIM)
        act_rows = starts[:, None] + np.arange(SPAN)[None]                          # (B, 20)
        actions = torch.from_numpy(ACTIONS_Z[act_rows]).view(len(starts), N_STEPS_CLIP, FRAMESKIP * ACTION_DIM)
        return frames.to(DEVICE), actions.to(DEVICE)

    def close(self):
        self.f.close()


def loss_fn(model, frames, actions):
    """Teacher-forced next-token MSE on the visual dims, as the reference's z_loss (without proprio)."""
    act_emb = model.extra_encoders["action"](actions)                               # (B, 4, 10)
    act_tiled = act_emb[:, :, None].expand(-1, -1, frames.shape[2], -1)
    emb = torch.cat([frames, act_tiled], dim=-1)                                    # (B, 4, tokens, 394)
    pred = model.predict(emb[:, :HISTORY])                                          # (B, 3, tokens, 394)
    loss = F.mse_loss(pred[..., :FEAT_DIM], emb[:, 1:, :, :FEAT_DIM])
    if not torch.isfinite(loss):
        raise RuntimeError(f"non-finite loss {loss.item()}: frames finite={torch.isfinite(frames).all().item()}, "
                           f"actions finite={torch.isfinite(actions).all().item()}")
    return loss


@torch.no_grad()
def validate(model, feats, rng, n_batches=20):
    model.eval()
    losses = []
    for _ in range(n_batches):
        starts = rng.choice(clips["val"], size=min(BATCH, len(clips["val"])), replace=False)
        losses.append(loss_fn(model, *feats.clip_batch(starts)).item())
    model.train()
    return float(np.mean(losses))

## 7. Training

One variant after another. Every `CKPT_EVERY` steps the checkpoint folder is written (`weights.pt`,
`config.json`, `meta.json`, plus `train_state.pt` for resuming), and the curve goes to
`results/training/<run>.csv`. On Colab every checkpoint is also copied to Drive right away, so a
disconnected session loses at most `CKPT_EVERY` steps, and a rerun resumes from the last checkpoint
(it restores from Drive if the session disk is empty). At each milestone in `MILESTONES` a snapshot
folder `<run>_step<N>` is kept, so the variants can be compared at equal budgets whatever step the
run reaches. After 100 steps the projected time for the variant is printed.

In [ ]:
def run_name(variant):
    return f"dwt_{TAG}_{variant}"


def backup(folder):
    """Copy a checkpoint folder to Drive (no-op outside Colab)."""
    if DRIVE_DIR is not None:
        shutil.copytree(folder, DRIVE_DIR / "checkpoints" / folder.name, dirs_exist_ok=True)


def restore(name):
    """Bring a checkpoint folder back from Drive if the session disk lost it."""
    if DRIVE_DIR is not None and not (CKPT_DIR / name).exists() and (DRIVE_DIR / "checkpoints" / name).exists():
        shutil.copytree(DRIVE_DIR / "checkpoints" / name, CKPT_DIR / name)
        print(f"{name}: restored from Drive")


def save_checkpoint(model, opt, step, variant, val_loss, train_loss):
    name = run_name(variant)
    swm.wm.utils.save_pretrained(model, name, config=OmegaConf.create(model_config(variant, action_dim=ACTION_DIM)))
    folder = CKPT_DIR / name
    torch.save({"optimizer": opt.state_dict(), "step": step}, folder / "train_state.pt")
    (folder / "meta.json").write_text(json.dumps({
        "variant": variant, "tokens_per_frame": TOKENS[variant], "features": TAG, "train_episodes": int(len(episodes["train"])),
        "val_episodes": int(n_val), "steps_done": step, "steps_planned": STEPS, "batch": BATCH, "lr": LR,
        "action_mean": ACTION_MEAN.tolist(), "action_std": ACTION_STD.tolist(),
        "val_mse": val_loss, "train_mse": train_loss, "gpu": GPU, "seed": SEED}, indent=1))
    backup(folder)
    if DRIVE_DIR is not None:                                  # the training curve goes with every checkpoint
        (DRIVE_DIR / "training").mkdir(exist_ok=True)
        shutil.copy(LOG_DIR / f"{name}.csv", DRIVE_DIR / "training" / f"{name}.csv")
    if step in MILESTONES:
        snapshot = CKPT_DIR / f"{name}_step{step}"
        shutil.copytree(folder, snapshot, dirs_exist_ok=True)
        (snapshot / "train_state.pt").unlink()                 # snapshots are for evaluation, not resuming
        backup(snapshot)


def train_variant(variant):
    torch.manual_seed(SEED)
    rng = np.random.default_rng(SEED)
    feats = Features(FILES[variant])
    model = build_model(variant, action_dim=ACTION_DIM).to(DEVICE).train()
    params = trainable_parameters(model)
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=WEIGHT_DECAY)

    folder, step = CKPT_DIR / run_name(variant), 0
    restore(run_name(variant))
    if (folder / "train_state.pt").exists():
        state = torch.load(folder / "train_state.pt", map_location=DEVICE)
        model.load_state_dict(torch.load(folder / "weights.pt", map_location=DEVICE))
        opt.load_state_dict(state["optimizer"])
        step = state["step"]
        rng = np.random.default_rng(SEED + step)
        if not all(torch.isfinite(p).all() for p in params):
            raise RuntimeError(f"{folder} holds non-finite weights; delete it (and its copy on Drive) and rerun")
        print(f"{variant}: resuming at step {step}")
    if step >= STEPS:
        print(f"{variant}: already trained for {STEPS} steps, skipping")
        feats.close()
        return

    log_path = LOG_DIR / f"{run_name(variant)}.csv"
    if DRIVE_DIR is not None and not log_path.exists() and (DRIVE_DIR / "training" / log_path.name).exists():
        shutil.copy(DRIVE_DIR / "training" / log_path.name, log_path)      # continue the curve after a resume
    log = log_path.open("a", newline="")
    writer = csv.writer(log)
    if log_path.stat().st_size == 0:
        writer.writerow(["variant", "step", "train_mse", "val_mse", "elapsed_s"])

    t0, recent = time.perf_counter(), []
    val_loss = validate(model, feats, rng)
    print(f"{variant}: {TOKENS[variant]} tokens | {sum(p.numel() for p in params) / 1e6:.1f} M trainable | "
          f"val MSE before training {val_loss:.4f} | features {'in RAM' if feats.in_ram else 'from disk'}")
    while step < STEPS:
        starts = rng.choice(clips["train"], size=BATCH, replace=False)
        loss = loss_fn(model, *feats.clip_batch(starts))
        opt.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(params, GRAD_CLIP)
        opt.step()
        step += 1
        recent.append(loss.item())
        if step == 100:
            per_step = (time.perf_counter() - t0) / 100
            print(f"  {per_step * 1000:.0f} ms per step -> about {per_step * STEPS / 60:.0f} min for {STEPS} steps")
        if step % EVAL_EVERY == 0 or step == STEPS:
            val_loss = validate(model, feats, rng)
            train_loss = float(np.mean(recent)); recent = []
            writer.writerow([variant, step, f"{train_loss:.6f}", f"{val_loss:.6f}", f"{time.perf_counter() - t0:.0f}"]); log.flush()
            print(f"  step {step:6d} | train {train_loss:.4f} | val {val_loss:.4f} | {(time.perf_counter() - t0) / 60:.1f} min")
        if step % CKPT_EVERY == 0 or step == STEPS:
            save_checkpoint(model, opt, step, variant, val_loss, train_loss)
    log.close()
    feats.close()
    return val_loss


SUMMARY = {}
for variant in VARIANTS:
    SUMMARY[variant] = train_variant(variant)

## 8. Summary

Validation MSE per variant (lower is better; all variants predict 384-dim tokens, so the values are comparable).

In [ ]:
import pandas as pd

rows = []
for variant in VARIANTS:
    meta = json.loads((CKPT_DIR / run_name(variant) / "meta.json").read_text())
    rows.append({"variant": variant, "tokens": meta["tokens_per_frame"], "steps": meta["steps_done"],
                 "train MSE": round(meta["train_mse"], 4), "val MSE": round(meta["val_mse"], 4)})
table = pd.DataFrame(rows).set_index("variant")

if DRIVE_DIR is not None:
    shutil.copytree(LOG_DIR, DRIVE_DIR / "training", dirs_exist_ok=True)
    print("checkpoints and logs are on Drive under", DRIVE_DIR)
    print(sorted(p.name for p in (DRIVE_DIR / "checkpoints").iterdir()))
table

## Notes

- `val MSE` is teacher-forced one-step error on held-out episodes: a cheap proxy, not planning success.
  Notebook 05 measures success and planning time.
- The `mean` variant has no spatial information by construction and is expected to be the weakest.
- To train longer, raise `STEPS` and rerun: training resumes from the saved state.
- Overnight on free Colab: keep the browser tab open and the computer awake (Colab has no background
  execution on the free tier). If the session still drops, rerun from the top: it resumes.